# 02. Inline claim-based authz

Same shared API key as pattern 1, but the agent now does something with the
user's identity. It fetches the user's JWT, reads the claims out of it, and
constructs scoped tool calls based on what the claims say.

The tool still has no idea who's asking (it just sees `X-API-Key`), but
at least the call going out is narrower than "give me everything".

## The gap from pattern 1

In pattern 1 the agent had no information to act on. The LLM was the only
thing keeping alice from seeing dave's data, and the LLM is not a security
boundary.

This pattern fixes the *information* problem. The agent now knows the user's
role, department, and reporting structure, and uses that information to
scope the call before it goes out. Authz is still done in the agent, but
with structured data instead of just prompt-following.

In [ ]:
from shared.agent import Agent
from shared.auth import InlineClaimAuth, fetch_user_jwt, decode_jwt
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw, show_token
from shared.config import EXPENSE_SERVICE_URL

strategy = InlineClaimAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

## What's in the JWT the agent reads

Before running the agent, look at what `InlineClaimAuth` will see when it
fetches alice's user JWT. The custom claims on this token are the input it
uses to make scoping decisions.

In [ ]:
alice_jwt = fetch_user_jwt("alice")
show_token(alice_jwt, label="alice user JWT (what InlineClaimAuth reads)")

## Same prompt, three users. Watch where the scoping breaks down.

In [ ]:
result_alice = run_as("alice", "What expenses do I have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_bob = run_as("bob", "What expenses do my team have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_dave = run_as("dave", "Show me all the expenses across the company.", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Tradeoffs

- **Authz lives in the agent.** Specifically, in `InlineClaimAuth.prepare()`,
  which has hand-coded rules for which query parameters to add for which
  roles. Look at the source: every new role, every new tool, every new field
  requires editing that code.
- **The tool still sees no real user.** Service still reports `method=api_key`.
  The narrowing happens before the call goes out, not on the service side.
- **Still no cryptographic proof of identity.**
- **Partial least privilege.** The agent can narrow what it asks for, but it
  could ask for anything if it wanted to (or if it had a bug).
- **The gap that breaks the demo for alice:** look at what just happened.
  The agent could narrow the call for bob (a manager: it added
  `?department=engineering`) and didn't need to narrow for dave (admin). For
  alice the agent had no useful query parameter to add, because the service
  doesn't accept `?user_id=alice` from an api-key caller. So alice's call
  goes out unfiltered and her "expenses" include everyone's. **This is the
  pedagogical point.** Doing authz in the agent only works as well as the
  API surface lets you express it.
- **Audit trail:** logs still say "agent called expense-service with the
  api key". The fact that the agent internally knew it was acting for alice
  is invisible to the service.

Real codebases that do this kind of inline scoping tend to grow a thicket
of role-checks across the codebase. Every tool has slightly different
rules, every new feature requires touching auth logic, and review burden
goes up linearly with feature count.

## What's still missing

The agent can use claims to narrow some calls. But:

1. The logic lives in agent code, not in policy. Adding a new tool means
   editing `InlineClaimAuth`.
2. The narrowing only works as far as the tool's API surface allows. Alice's
   case is the proof.
3. The tool still trusts whoever has the API key.

Notebook 03 keeps the same shape (agent has the JWT, tool gets the API key)
but moves the policy logic out of agent code into a centralized policy
engine (OPA). That fixes problem 1: the rules become declarative and
auditable. It does not fix problems 2 and 3. Those are what patterns 4 to
8 are for.